# Lab 4, Silver sales orders (SCD Type 1)

Sales orders are a fact table, not a dimension, so this uses SCD Type 1 instead of Type 2. If an order gets corrected later I just want the corrected value, not a growing pile of history for every little fix. That's the main difference from the customers notebook.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("catalog", "lab4", "Catalog")
catalog = dbutils.widgets.get("catalog")

First let me just look at the raw nested shape again before flattening anything. ordered_products is an array of structs, and each struct even has its own nested promotion_info struct inside it.

In [0]:
orders_bronze = spark.table(f"{catalog}.bronze.brz_sales_orders")
orders_bronze.select("order_number", "ordered_products").limit(3).show(truncate=80)

## Flattening to one row per line item

explode() turns the array into one row per product, and the order level columns just get repeated across each line item, which is correct, a 3 item order becomes 3 rows.

order_datetime comes in as a Unix epoch string, so I'm casting it properly to a timestamp instead of leaving it as text. Using try_cast here instead of a plain cast, since a plain cast blew up on some empty string values when I first tried this (number_of_line_items has the same issue, some rows are just empty strings). try_cast just returns null on bad input instead of failing the whole job.

Now that bronze is fully string-typed, the same problem exists for every other numeric field too (order_number, unit_price, quantity, promo_id, promo_discount), so try_cast is applied across all of them here rather than just the two I originally ran into trouble with.

In [0]:
orders_flat = (
    orders_bronze
    .withColumn("line_item", F.explode("ordered_products"))
    .select(
        F.expr("try_cast(order_number AS BIGINT)").alias("order_number"),
        F.col("customer_id"),
        F.col("customer_name"),
        F.expr("try_cast(try_cast(order_datetime AS BIGINT) AS TIMESTAMP)").alias("order_datetime"),
        F.expr("try_cast(number_of_line_items AS INT)").alias("number_of_line_items"),
        F.col("line_item.id").alias("product_id"),
        F.col("line_item.name").alias("product_name"),
        # kept as BIGINT here to match the current silver table declaration below - 04 widens this
        # to DOUBLE later on purpose, that's the lab's "widening a column" exercise, so this needs to
        # stay narrow for that demo to actually have something to widen
        F.expr("try_cast(line_item.price AS BIGINT)").alias("unit_price"),
        F.expr("try_cast(line_item.qty AS BIGINT)").alias("quantity"),
        F.col("line_item.unit").alias("unit"),
        F.col("line_item.curr").alias("currency"),
        F.expr("try_cast(line_item.promotion_info.promo_id AS BIGINT)").alias("promo_id"),
        F.expr("try_cast(line_item.promotion_info.promo_disc AS DOUBLE)").alias("promo_discount"),
        # revenue itself uses full DOUBLE precision regardless of unit_price's storage type above,
        # so a future fractional price wouldn't silently lose cents in this calculation
        F.round(
            F.expr("try_cast(line_item.price AS DOUBLE)") * F.expr("try_cast(line_item.qty AS BIGINT)"),
            2
        ).alias("line_revenue"),
    )
)

print(f"Flattened: {orders_flat.count()} line item rows (from {orders_bronze.count()} orders)")
orders_flat.limit(5).show(truncate=40)

Since try_cast can silently turn bad values into null, I want to actually know how many nulls that produced instead of just hoping it's fine.

The cell right after this actually quarantines on the merge key specifically (order_number/product_id), so this check here is just a broader look at null rates across the other cast fields, not a filtering step by itself.

In [0]:
orders_flat.selectExpr(
    "COUNT(*) AS total_rows",
    "SUM(CASE WHEN order_datetime IS NULL THEN 1 ELSE 0 END) AS null_order_datetime",
    "SUM(CASE WHEN number_of_line_items IS NULL THEN 1 ELSE 0 END) AS null_number_of_line_items"
).show()

## Quarantining rows with a broken merge key
`order_number` and `product_id` together are the merge key for this table. `product_id` is a string so it doesn't need a numeric cast, but it does need to actually be present. `order_number` does need a numeric cast, and now that bronze is all strings, a malformed value there would previously have just silently become `null` and merged as if it were a legitimate key. Splitting those out instead.

In [0]:
orders_quarantine = (
    orders_flat
    .filter("order_number IS NULL OR product_id IS NULL OR product_id = ''")
    .withColumn(
        "rejection_reason",
        F.when(F.col("order_number").isNull(), F.lit("order_number did not cast to BIGINT"))
         .otherwise(F.lit("product_id missing"))
    )
    .withColumn("quarantined_at", F.current_timestamp())
)

orders_clean = orders_flat.filter("order_number IS NOT NULL AND product_id IS NOT NULL AND product_id != ''")

quarantine_count = orders_quarantine.count()
print(f"{quarantine_count} row(s) failed the merge-key check and were quarantined")

if quarantine_count > 0:
    (orders_quarantine.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(f"{catalog}.silver.slv_sales_order_lines_quarantine")
    )

In [0]:
orders_clean.createOrReplaceTempView("orders_flat_vw")

## Checking for duplicates

The real key for a fact table like this isn't order_number by itself, it's order_number plus product_id, since one order can legitimately have several products in it. Checking against the wrong key would give a false positive here.

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_number, product_id) AS distinct_line_items,
    COUNT(*) - COUNT(DISTINCT order_number, product_id) AS duplicate_rows
FROM orders_flat_vw

## Dedup

Same pattern as the customers notebook. For ties I'm keeping the row with the higher line_revenue, same logic as before, keep the more complete looking row.

In [0]:
from pyspark.sql.window import Window

orders_window = Window.partitionBy("order_number", "product_id").orderBy(F.desc("line_revenue"))

orders_dedup = (
    orders_clean
    .withColumn("row_num", F.row_number().over(orders_window))
    .filter("row_num = 1")
    .drop("row_num")
)

print(f"Deduped: {orders_dedup.count()} line item rows")

## Creating the silver fact table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS lab4.silver.slv_sales_order_lines (
    order_number BIGINT,
    product_id STRING,
    customer_id STRING,
    customer_name STRING,
    order_datetime TIMESTAMP,
    number_of_line_items INT,
    product_name STRING,
    unit_price BIGINT,
    quantity BIGINT,
    unit STRING,
    currency STRING,
    promo_id BIGINT,
    promo_discount DOUBLE,
    line_revenue DOUBLE,
    silver_updated_at TIMESTAMP
)
USING DELTA

## Merge, SCD Type 1 this time

Much simpler than the customers version. Matched rows just get overwritten with the latest values, no is_current or effective_end bookkeeping needed, that's really the whole difference between Type 1 and Type 2.

In [0]:
def run_scd1_merge(source_df, target_table_name, key_cols):
    target = DeltaTable.forName(spark, target_table_name)

    match_condition = " AND ".join([f"target.{k} = source.{k}" for k in key_cols])

    (target.alias("target")
        .merge(source_df.alias("source"), match_condition)
        .whenMatchedUpdate(
            set={
                **{c: f"source.{c}" for c in source_df.columns if c not in key_cols},
                "silver_updated_at": "current_timestamp()",
            }
        )
        .whenNotMatchedInsert(
            values={
                **{c: f"source.{c}" for c in source_df.columns},
                "silver_updated_at": "current_timestamp()",
            }
        )
        .execute()
    )


run_scd1_merge(orders_dedup, f"{catalog}.silver.slv_sales_order_lines", ["order_number", "product_id"])

spark.sql(f"SELECT COUNT(*) AS total_rows FROM {catalog}.silver.slv_sales_order_lines").show()

## Idempotency check

Rerunning the same merge with the exact same data. Row count shouldn't move at all.

In [0]:
before_count = spark.table(f"{catalog}.silver.slv_sales_order_lines").count()

run_scd1_merge(orders_dedup, f"{catalog}.silver.slv_sales_order_lines", ["order_number", "product_id"])

after_count = spark.table(f"{catalog}.silver.slv_sales_order_lines").count()

print(f"Before rerun: {before_count} rows")
print(f"After rerun: {after_count} rows")
print("Idempotent, good" if before_count == after_count else "Not idempotent, need to check this")

## Quick sanity check

Just want to see if revenue by product actually looks like real, coherent data, not just correctly shaped rows that don't mean anything.

In [0]:
%sql
SELECT product_name, SUM(line_revenue) AS total_revenue, SUM(quantity) AS total_units
FROM lab4.silver.slv_sales_order_lines
GROUP BY product_name
ORDER BY total_revenue DESC
LIMIT 10